In [1]:
import pandas as pd
import numpy as np

market = pd.read_csv("../data/market_master.csv")

market["Date"] = pd.to_datetime(market["Date"])
market = market.set_index("Date")

In [2]:
#Basic Position Logic
market["position"] = np.where(
    market["mom_20_norm"] > 0, 1,
    np.where(market["mom_20_norm"] < 0, -1, 0)
)
#Position Change (Trades)
market["position_change"] = market["position"].diff().abs().fillna(0)

#Transaction Cost
cost_per_trade = 0.0005   # 5 basis points

market["cost"] = market["position_change"] * cost_per_trade

#Strategy Returns
market["strategy_ret"] = (
    market["position"].shift(1) * market["nifty_ret"]
    - market["cost"]
)

market = market.dropna(subset=["strategy_ret"])

In [4]:
market.to_csv("../backtesting/maket_backtest_v1.csv")